# Gradient methods

Gradient optimisation is a family of iterative methods used to find an optimum (minimum or maximum) of an objective function. It repeatedly updates the function parameters in the direction opposite to the gradient. The gradient is a vector that, at every point, indicates the direction of the steepest increase of the function. Moving in the opposite direction therefore allows us to "descend" towards progressively lower function values. The process is repeated until a point is reached where the gradient is close to zero, which usually indicates a local minimum.

One of the most important hyperparameters in gradient optimisation is the **learning rate**. It determines the size of the step taken in the direction opposite to the gradient at each iteration. If the learning rate is too large, the algorithm may overshoot the minimum or even diverge; if it is too small, convergence may become extremely slow. Selecting an appropriate learning rate can itself be treated as a separate optimisation problem (hence the term *hyperparameter*) or handled using methods that estimate a suitable value approximately.

The following example shows the cost function $f(x)=x^4-4x^2+2x+1$ over the domain $[-3,3]$, which contains both a local and a global minimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def target_fn(x: float | np.ndarray) -> float | np.ndarray:
    return x**4 - 4*x**2 + 2*x + 1

In [ ]:
x = np.linspace(-3, 3, 100)
y = target_fn(x)

In [ ]:
x_min_global = -1.61803
y_min_global = target_fn(x_min_global)

x_min_local = 1.3468
y_min_local = target_fn(x_min_local)

In [ ]:
plt.plot(x, y)
plt.scatter(x_min_local, y_min_local, color="red", label="Local minimum")
plt.scatter(x_min_global, y_min_global, color="green", label="Global minimum")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Function with a local and a global minimum")
plt.legend()

plt.show()

## Computing gradients

The general gradient-descent algorithm is based on the fact that, for any point (parameter vector) of the cost function, the direction opposite to the gradient ($-\nabla f(x)$) points towards the steepest local decrease. Put more simply, the gradient indicates the direction of the fastest increase of the function at a given point; moving in the opposite direction therefore leads towards a minimum. A function optimised with gradient methods must satisfy two key properties: it should be continuous and differentiable at every point considered.

### Gradient-optimisation algorithm

1. We seek a point $x^*\in\mathbb{R}^n$ that minimises the cost function $f\colon\mathbb{R}^n\rightarrow\mathbb{R}$: $x^*=\text{argmin}_{x\in\mathbb{R}^n}f(x)$.
2. Initialise:
   1. a starting point $x^{(0)}\in\mathbb{R}^n$,
   2. the learning-rate hyperparameter $\alpha$,
   3. a convergence tolerance $\epsilon$ and/or a maximum number of iterations $K$.
3. For $k=0,\ldots,K$:
   1. at the current point $x^{(k)}$, compute the scalar derivative for a one-parameter function, $J'(f(x))$, or, more generally, the gradient vector: $\nabla f(x^{(k)})=\left[\frac{\partial f(x^{(k)})}{\partial x_1},\frac{\partial f(x^{(k)})}{\partial x_2},\ldots,\frac{\partial f(x^{(k)})}{\partial x_n}\right]^T$;
   2. check the stopping condition (convergence), i.e. whether $x^*\approx x^{(k)}$:
      1. the gradient norm is small: $\lVert\nabla f(x^{(k)})\rVert<\epsilon$, and/or
      2. the change in cost-function values at neighbouring points is small: $|f(x^{(k+1)})-f(x^{(k)})|<\epsilon$, and/or
      3. the iteration limit has been reached: $k\geq K$;
   3. update the parameters in the direction opposite to the gradient: $x^{(k+1)}=x^{(k)}-\alpha_k\cdot\nabla f(x^{(k)})$.

### Simulating gradient optimisation with different learning-rate settings

The example below extends the previous visualisation with several searches for minima of the cost function using gradients and different learning-rate values.

Define the derivative of the one-parameter cost function, $J'(x)=4x^3-8x+2$, as a separate function.

In [ ]:
def df(x: float) -> float:
    return 4 * x**3 - 8 * x + 2

Simulate gradient descent as a trajectory of successive function values.

In [ ]:
def gradient_descent(x_start: float, lr: float, steps: int) -> np.ndarray:
    trajectory = [x_start]
    x = x_start
    for _ in range(steps):
        x = x - lr * df(x)
        trajectory.append(x)

    return np.array(trajectory)

Simulation settings for different starting points and learning-rate values.

In [ ]:
scenarios = [
    {
        "x0": -2.3,
        "lr": 0.04,
        "color": "green",
        "label": "1. Correct convergence (global minimum)",
    },
    {
        
        "x0": 1.0,
        "lr": 0.05,
        "color": "orange",
        "label": "2. Trapped in a local minimum",
    },
    {
        "x0": -2.1,
        "lr": 0.1,
        "color": "red",
        "label": "3. Overshooting the minimum (step too large)",
    },
]

Visualise the complete simulation.

In [ ]:
x_vals = np.linspace(-2.5, 2.3, 500)
y_vals = target_fn(x_vals)

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(x_vals, y_vals, "k-", linewidth=2, alpha=0.7, label="Function $f(x)$")

# Draw the trajectory and gradient vectors for each scenario
for sc in scenarios:
    traj_x = gradient_descent(sc["x0"], sc["lr"], steps=10)
    traj_y = target_fn(traj_x)

    # Draw points and the line connecting successive steps
    plt.plot(
        traj_x,
        traj_y,
        "o--",
        color=sc["color"],
        linewidth=1.5,
        markersize=6,
        label=sc["label"],
    )

    # Draw gradient vectors (arrows) for the first two steps
    for i in range(2):
        x_i = traj_x[i]
        y_i = traj_y[i]
        dx = -sc["lr"] * df(x_i)  # Step in the direction opposite to the gradient
        dy = target_fn(x_i + dx) - y_i

        plt.arrow(
            x_i,
            y_i,
            dx,
            dy,
            head_width=0.1,
            head_length=0.3,
            fc=sc["color"],
            ec=sc["color"],
            length_includes_head=True,
            alpha=0.8,
        )
# Annotate minima
plt.scatter(
    [x_min_local, x_min_global],
    [target_fn(x_min_local), target_fn(x_min_global)],
    color="black",
    s=100,
    zorder=5,
    marker="X",
)
plt.text(
    x_min_local, target_fn(x_min_local) - 1, "Local min.", horizontalalignment="center"
)
plt.text(
    x_min_global, target_fn(x_min_global) - 1, "Global min.", horizontalalignment="center"
)

plt.title(
    "Effect of the starting point and learning rate on gradient optimisation"
)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.xlim(-2.5, 2.5)
plt.ylim(-7, 8)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(loc="upper center")

plt.show()

## Popular applications of gradient optimisation

Gradient optimisation is widely used whenever the objective function contains many parameters.

1. Machine learning: linear/nonlinear/logistic regression, SVMs, gradient boosting, and neural networks (backpropagation) through minimisation of an error function.
2. Engineering: optimisation of design parameters.
3. Finance: investment-portfolio optimisation and risk analysis.

## Gradient optimisation in practice

### Optimising a polynomial function

In [ ]:
from scipy.optimize import minimize

Cost function: $f(x,y,z,v,t)=(x-3)^5+(y+10)^4+z^3-(v+40)^2+4.5t$

In [ ]:
def cost_fn(params: list) -> float:
    x, y, z, v, t = params
    return (x - 3) ** 5 + (y + 10) ** 4 + z ** 3 - (v + 40) ** 2 + 4.5 * t

Function for computing the gradient of the cost function: $\nabla f=\left[\frac{\partial f}{\partial x},\frac{\partial f}{\partial y},\frac{\partial f}{\partial z},\frac{\partial f}{\partial v},\frac{\partial f}{\partial t}\right]$:
- $\frac{\partial f}{\partial x}=5(x-3)^4$
- $\frac{\partial f}{\partial y}=4(y+10)^3$
- $\frac{\partial f}{\partial z}=3z^2$
- $\frac{\partial f}{\partial v}=-2(v+40)$
- $\frac{\partial f}{\partial t}=4.5$

In [ ]:
def grad_cost_fn(params: list) -> np.ndarray:
    x, y, z, v, t = params

    df_dx = 5 * (x - 3) ** 4
    df_dy = 4 * (y + 10) ** 3
    df_dz = 3 * (z**2)
    df_dv = -2 * (v + 40)
    df_dt = 4.5

    return np.array([df_dx, df_dy, df_dz, df_dv, df_dt])

Define the starting point and the stopping condition.

In [ ]:
initial_point = [0., 0., 0., 0., 0.]
epsilon = 1e-4

Search for the minimum of the cost function using the `BFGS` method.

In [ ]:
result = minimize(cost_fn, initial_point, jac=grad_cost_fn, method="BFGS", tol=epsilon)

Minimum function value

In [ ]:
result.fun.item()

Optimal parameters

In [ ]:
result.x

### Optimising a logistic-regression model

The purpose of this example is to demonstrate gradient optimisation in a practical machine-learning setting. Logistic regression will be used to classify a linearly separable two-dimensional data set. The example illustrates how, over successive iterations updating the parameters of the mathematical model, both the value of the cost function and classification accuracy change.

A logistic-regression model estimates the probability ($p$) that a given object belongs to the positive decision class by transforming a linear combination of attributes ($x$) and parameters ($w$) with the sigmoid function $\sigma(z)$: $\hat{y}=p=\sigma(w^Tx+b)=\frac{1}{1+e^{-(w^Tx+b)}}$.

The quality of the logistic-regression model will be measured using **binary cross-entropy (BCE)**, which applies a logarithmic penalty for incorrect classifications:
$BCE(y,\hat{y})=-\left[y\ln(\hat{y})+(1-y)\ln(1-\hat{y})\right]$. For the whole data set this can be generalised as $J(\vec{y},\vec{\hat{y}})=-\frac{1}{N}\sum_{i=1}^{N}BCE(y_i,\hat{y}_i)$, where:
- $y$ — target class value,
- $\hat{y}$ — class probability returned by the model,
- $N$ — number of objects in the data set.

Here, BCE acts as the cost function. Minimising classification error gradually leads the model towards better predictions.

In [ ]:
from IPython.display import HTML
import matplotlib.animation as animation
import torch
import torch.nn as nn

In [ ]:
np.random.seed(42)

Generate a 40-element data set in which class 0 lies in the lower-left region and class 1 in the upper-right region.

In [ ]:
X_raw = np.random.uniform(-4, 4, (200, 2))
margin = 0.5

In [ ]:
x0 = X_raw[X_raw[:, 0] + X_raw[:, 1] < -margin][:40]
x1 = X_raw[X_raw[:, 0] + X_raw[:, 1] > margin][:40]

In [ ]:
plt.scatter(x0[:, 0], x0[:, 1], color="blue", label="Klasa 0",)
plt.scatter(x1[:, 0], x1[:, 1], color="red", label="Klasa 1")
plt.show()

Prepare the data set in the format expected by PyTorch.

In [ ]:
X_data = np.vstack((x0, x1))
y_data = np.hstack((np.zeros(40), np.ones(40)))

X_tensor = torch.tensor(X_data, dtype=torch.float32)
y_tensor = torch.tensor(y_data, dtype=torch.float32).unsqueeze(1)

Define a single-layer neural network with a logistic output function (equivalent to a simple logistic-regression model).

In [ ]:
class LogisticRegressionModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(2, 1)

    def forward(self, x: float) -> torch.Tensor:
        return torch.sigmoid(self.linear(x))

Create the model and define binary cross-entropy as the cost function.

In [ ]:
K = 40
lr = 0.2

In [ ]:
model = LogisticRegressionModel()
criterion = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

Optimisation loop

In [ ]:
history_loss = []
history_weights = []
history_bias = []

In each iteration:
- generate the output, i.e. decision-class predictions for every point;
- compute the cost-function value;
- compute the gradients;
- update the parameters.

In [ ]:
for epoch in range(K):
    optimizer.zero_grad()
    outputs = model(X_tensor)  # outputs from logistic regression
    loss = criterion(outputs, y_tensor)  # cost-function value
    loss.backward()  # gradient values
    optimizer.step()  # parameter update

    history_loss.append(loss.item())
    history_weights.append(model.linear.weight.data.numpy().copy())
    history_bias.append(model.linear.bias.data.numpy().copy())

Visualisation

In [ ]:
def update(epoch: int) -> None:
    ax1.clear()
    ax2.clear()

    # restore weights for the given iteration
    w = history_weights[epoch][0]
    b = history_bias[epoch][0]

    # compute probabilities on the grid for the background
    Z = torch.sigmoid(grid_tensor @ torch.tensor(w) + b).numpy()
    Z = Z.reshape(xx.shape)

    # plot 1: decision boundary and points
    ax1.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.6)
    ax1.contour(
        xx, yy, Z, levels=[0.5], colors="k", linewidths=2
    )  # p=0.5 boundary
    ax1.scatter(
        x0[:, 0], x0[:, 1], color="blue", edgecolor="k", label="Class 0"
    )
    ax1.scatter(
        x1[:, 0], x1[:, 1], color="red", edgecolor="k", label="Class 1"
    )
    ax1.set_title(f"Epoch {epoch+1}/{K} - Decision boundary")
    ax1.set_xlabel("$X_1$")
    ax1.set_ylabel("$X_2$")
    ax1.legend(loc="upper left")

    # plot 2: decrease of the cost function
    ax2.plot(
        range(1, epoch + 2), history_loss[: epoch + 1], "r-o", linewidth=2
    )
    ax2.set_xlim(0, K + 1)
    ax2.set_ylim(0, max(history_loss) * 1.1)
    ax2.set_title(f"Cost (BCE loss): {history_loss[epoch]:.4f}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.grid(True, linestyle="--", alpha=0.5)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200)
)
grid_tensor = torch.tensor(
    np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32
)

ani = animation.FuncAnimation(
    fig, update, frames=K, interval=250, repeat=False
)
plt.tight_layout()
plt.close()
HTML(ani.to_jshtml())